In [1]:
using LowLevelFEM

In [2]:
gmsh.initialize()

In [3]:
structured_rect_mesh(lx=2, order=2)

In [4]:
material = Material("surface")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2)

Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv)

In [5]:
pres0 = BoundaryCondition("rightbottom", problem=Pp, p=0)

BoundaryCondition("rightbottom", Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp), Dict{Symbol, Union{Function, Number, ScalarField}}(:p => 0))

In [6]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0)

BoundaryCondition("bottom", Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Dict{Symbol, Union{Function, Number, ScalarField}}(:vy => 0, :vx => 0))

In [7]:
load_v = LoadCondition("surface", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp])

SystemVector([0.0002777777777777773, 0.0, 0.0002777777777777746, 0.0, 0.0002777777777777775, 0.0, 0.00027777777777777767, 0.0, 0.0005555555555555548, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], nothing, Problem[Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)], [0, 1722])

In [8]:
μ = 1.0

# stabilization
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 1e-6          # pressure Laplacian (mesh dependent)

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)
D = ∫(Div(Pv) ⋅ Div(Pv) * γ)

# alternative way
AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)))

B = ∫(Div(Pv) ⋅ Pp)
C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

K = SystemMatrix([A+D B';
    B -C])

sparse([1, 2, 9, 10, 47, 48, 219, 220, 239, 240  …  1722, 1725, 1774, 1784, 1785, 1804, 2013, 2563, 2581, 2583], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583], [1.8666666666666698, 1.0000000000000044, 0.02222222222222278, -5.551115123125783e-17, -1.1111111111111125, -1.7763568394002505e-15, -0.2222222222222221, -3.0531133177191805e-16, -0.08888888888888558, 2.220446049250313e-16  …  2.42861286636753e-17, 0.7111111111111112, 0.7111111111111188, 2.133333333333322, 0.7111111111111057, 2.133333333333322, 0.7111111111111028, 2.1333333333333373, 2.1333333333333457, -11.377777777777766], 2583, 2583)

In [9]:
K[:, :]

2583×2583 SparseArrays.SparseMatrixCSC{Float64, Int64} with 116825 stored entries:
⎡⣿⢟⣉⠙⠿⠷⠶⠶⢾⡋⠯⠽⠸⠸⠶⠆⠆⠆⠶⠰⠰⠰⠶⠆⡆⣋⣻⣏⠻⠷⢾⡿⠿⠷⠶⠶⠶⠶⢶⣋⎤
⎢⣇⠘⢿⣷⡶⠾⠿⠛⠛⣷⡶⠰⠰⠶⠆⠇⠇⠭⠽⠘⠘⠛⠃⠃⠃⠛⢹⢻⣷⠿⠛⣷⠶⠶⠿⠽⠛⠛⠛⠛⎥
⎢⢿⡇⣸⡏⠻⣦⡀⠀⠀⠉⠛⠻⠶⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣾⡿⣆⠀⠙⠷⣦⣄⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣿⠃⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠈⠉⠙⠛⠷⢶⣦⣤⣀⡀⠀⠀⢸⣿⠁⠹⣆⠀⠀⠈⠙⠻⣦⣄⡀⠀⎥
⎢⡾⠳⢿⣤⡄⠀⠀⠈⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠻⠶⢾⢿⣤⠀⠹⣦⠀⠀⠀⠀⠀⠙⠻⠶⎥
⎢⣏⡇⢘⡋⣿⡀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣘⣻⡀⠀⢿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⡂⢰⡆⢸⣧⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣲⣾⡇⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⎥
⎢⠸⠇⠬⠅⠀⢿⡆⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠯⠅⣧⠀⠀⠀⢿⣧⠀⠀⠀⠀⠀⎥
⎢⠨⠅⡍⡅⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠨⢭⠅⢻⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⎥
⎢⢘⡃⣓⠃⠀⠀⢹⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⢘⣛⠀⢸⡇⠀⠀⠀⠀⢻⣧⠀⠀⠀⎥
⎢⢐⡂⣶⠀⠀⠀⠈⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⢐⣲⠀⠈⣇⠀⠀⠀⠀⠀⢻⣧⠀⠀⎥
⎢⠸⠇⠭⠀⠀⠀⠀⠸⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠸⠯⠀⠀⣿⠀⠀⠀⠀⠀⠈⢻⣧⠀⎥
⎢⡬⢩⣭⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣬⣭⠀⠀⢸⡆⠀⠀⠀⠀⠀⠀⢻⣧⎥
⎢⡿⢾⣷⣒⣲⣶⣶⣶⣾⣗⣒⢲⢰⣰⡶⡆⡆⣆⣶⢰⢰⣰⡶⡆⡆⣿⣿⣿⣲⣶⣾⣗⣶⣶⣶⣶⣶⣶⣶⣿⎥
⎢⢿⡆⣽⡟⠻⢯⣅⡀⠀⠛⠛⠺⠾⠿⠥⣥⣥⣁⣀⣀⡀⠀⠀⠀⠀⠀⢸⣾⡿⣯⡀⠛⠿⠯⣭⣁⣀⠀⠀⠀⎥
⎢⣾⡷⢿⣤⣄⠀⠈⠙⠳⣦⣤⣄⠀⠀⠀⠀⠀⠀⠉⠉⠉⠙⠛⠛⠲⠶⢾⢿⣤⠈⠻⣦⡀⠀⠀⠈⠉⠛⠳⠶⎥
⎢⢿⡇⢸⡇⠹⣧⡀⠀⠀⠀⠉⠛⠿⣶⣤⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡿⡇⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣟⡇⠀⠙⣷⡀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣤⣀⠀⠀⠀⠀⠀⠀⢸⣿⠇⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⢸⡇⣿⠀⠀⠀⠈⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣦⣀⠀⠀⢸⣿⠀⠘⣧⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡼⢳⣿⠀⠀⠀⠀⠈⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣼⣿⠀⠀⢹⡆⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [10]:
v, p = solveField(K, F, support=[pres0, suppT, suppB])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.011297741328270555; 0.00629789381568244;;], [0.0], Int64[], 1, :v2D, Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv)), ScalarField(Matrix{Float64}[], [-0.0015188347053337822; 0.0; … ; 0.0015024261747974184; 0.0008788976825677312;;], [0.0], Int64[], 1, :scalar, Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)))

In [ ]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p")

openPostProcessor()

In [12]:
gmsh.finalize()